# 확장모형/부록모형 결과표 생성 노트북

## 1. 목적

이 노트북은 2024년 기업정보화통계조사 최종 분석용 데이터셋을 기준으로, 교수님 피드백을 반영한 확장모형을 추정하고 부록용 결과표를 생성한다.

이번 분석은 본문 주모형을 대체하는 것이 아니라, 정보화 담당 체계 세부 유형, 정보화 투자 범위, AI 구현 방식 및 활용 목적 변수를 보강했을 때 핵심 상호작용항의 방향과 유의성이 유지되는지 탐색적으로 확인하기 위한 부록용 확장분석이다.

## 2. 설정값 및 저장 옵션

`SAVE_OUTPUTS` 플래그 하나로 결과 파일 저장 여부를 관리한다.

- `SAVE_OUTPUTS = True`: 결과표 CSV/Markdown과 해석 메모를 `output/tables`에 저장한다.
- `SAVE_OUTPUTS = False`: 파일 저장 없이 노트북 화면에만 결과를 표시한다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

try:
    from IPython.display import display
except ImportError:
    display = None

# 저장 여부 단일 플래그
SAVE_OUTPUTS = False

CWD = Path.cwd()
if (CWD / 'working').exists():
    BASE_DIR = CWD
elif (CWD.parent / 'working').exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError('working 폴더가 있는 프로젝트 루트를 찾지 못했습니다.')

ANALYSIS_DIR = BASE_DIR / 'working' / 'analysis'
OUT_DIR = BASE_DIR / 'output' / 'tables'
if SAVE_OUTPUTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('OUT_DIR:', OUT_DIR)
print('SAVE_OUTPUTS:', SAVE_OUTPUTS)

BASE_DIR: /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터
OUT_DIR: /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터/output/tables
SAVE_OUTPUTS: False


## 3. 데이터 로드 및 변수 확인

최종 분석용 데이터셋은 `working/analysis` 폴더에서만 찾는다. 필수 변수, 확장모형 후보 변수, 강건성/참고 변수의 존재 여부와 결측 N을 확인한다.

In [2]:
patterns = ['*.csv', '*.xlsx', '*.parquet']
candidates = []
for priority, pattern in enumerate(patterns, start=1):
    for path in ANALYSIS_DIR.glob(pattern):
        candidates.append({
            'priority': priority,
            'file': path.name,
            'path': path,
            'modified_time': pd.Timestamp(path.stat().st_mtime, unit='s'),
            'size_bytes': path.stat().st_size,
        })

candidate_df = pd.DataFrame(candidates).sort_values(['priority', 'modified_time'], ascending=[True, False])
if candidate_df.empty:
    raise FileNotFoundError('working/analysis 안에서 csv/xlsx/parquet 분석 파일을 찾지 못했습니다.')

DATA_PATH = candidate_df.iloc[0]['path']
print('[working/analysis 데이터 후보]')
if display:
    display(candidate_df[['priority', 'file', 'modified_time', 'size_bytes']])
else:
    print(candidate_df[['priority', 'file', 'modified_time', 'size_bytes']].to_string(index=False))
print('\n사용 데이터 파일:', DATA_PATH.relative_to(BASE_DIR))

suffix = DATA_PATH.suffix.lower()
if suffix == '.csv':
    df = pd.read_csv(DATA_PATH)
elif suffix in ['.xlsx', '.xls']:
    df = pd.read_excel(DATA_PATH)
elif suffix == '.parquet':
    df = pd.read_parquet(DATA_PATH)
else:
    raise ValueError(f'지원하지 않는 파일 형식입니다: {suffix}')

total_n = len(df)
required_vars = ['effect_proc_improve', 'it_org_any', 'ai_use_sum', 'firm_size', 'industry']
model_a_vars = ['it_org_internal', 'it_org_mixed', 'it_org_outsource']
model_b_vars = ['it_invest_sum', 'it_invest_share', 'it_invest_share_std4', 'it_invest_high']
model_c_vars = ['ai_impl_sum', 'ai_purpose_sum']
reference_vars = ['effect_average', 'ai_use']
all_check_vars = required_vars + model_a_vars + model_b_vars + model_c_vars + reference_vars

variable_check = pd.DataFrame([
    {
        '변수명': var,
        '분류': '필수' if var in required_vars else 'A 후보' if var in model_a_vars else 'B 후보' if var in model_b_vars else 'C 후보' if var in model_c_vars else '참고',
        '존재 여부': var in df.columns,
        '유효 N': int(df[var].notna().sum()) if var in df.columns else np.nan,
        '결측 N': int(df[var].isna().sum()) if var in df.columns else np.nan,
        '고유값 예시': ', '.join(map(str, pd.Series(df[var].dropna().unique()).head(8).tolist())) if var in df.columns else '',
    }
    for var in all_check_vars
])
missing_required = variable_check.loc[(variable_check['분류'] == '필수') & (~variable_check['존재 여부']), '변수명'].tolist()
missing_candidates = variable_check.loc[(variable_check['분류'] != '필수') & (~variable_check['존재 여부']), '변수명'].tolist()

print('전체 N:', f'{total_n:,}')
print('\n[변수 존재 여부 및 결측 확인]')
if display:
    display(variable_check)
else:
    print(variable_check.to_string(index=False))

if missing_required:
    raise KeyError('필수 변수가 없어 확장모형을 구성할 수 없습니다: ' + ', '.join(missing_required))

[working/analysis 데이터 후보]


,priority,file,modified_time,size_bytes
0,1,nia_2024_analysis_total.csv,2026-04-27 19:01:15.774730444,1654412



사용 데이터 파일: working/analysis/nia_2024_analysis_total.csv
전체 N: 12,203

[변수 존재 여부 및 결측 확인]


,변수명,분류,존재 여부,유효 N,결측 N,고유값 예시
0,effect_proc_improve,필수,True,12203,0,"3, 4, 5, 2, 1"
1,it_org_any,필수,True,12203,0,"0, 1"
2,ai_use_sum,필수,True,12203,0,"1, 0, 2, 3, 4, 5, 6, 7"
3,firm_size,필수,True,12203,0,"1, 2, 3, 4"
4,industry,필수,True,12203,0,"1, 2, 3, 4, 5, 6, 7, 8"
5,it_org_internal,A 후보,True,12203,0,"0, 1"
6,it_org_mixed,A 후보,True,12203,0,"0, 1"
7,it_org_outsource,A 후보,True,12203,0,"0, 1"
8,it_invest_sum,B 후보,True,12203,0,"1, 2, 3, 4, 5, 6, 7, 8"
9,it_invest_share,B 후보,True,12203,0,"1.0, 4.0, 65.0, 2.0, 70.0, 35.0, 8.0, 50.0"


## 4. 확장모형 설계

모든 확장모형은 OLS로 추정하고, HC3 robust standard error를 사용한다. 모든 모형에는 `C(firm_size)`와 `C(industry)`를 포함한다.

In [3]:
notes = []

def has_vars(vars_):
    missing = [v for v in vars_ if v not in df.columns]
    return len(missing) == 0, missing

model_specs = []

a_vars = ['effect_proc_improve', 'it_org_internal', 'it_org_mixed', 'it_org_outsource', 'ai_use_sum', 'firm_size', 'industry']
use, miss = has_vars(a_vars)
model_specs.append({
    'model_name': 'A',
    'purpose': '정보화 담당 체계 유형별 확장모형',
    'outcome': 'effect_proc_improve',
    'formula': 'effect_proc_improve ~ it_org_internal * ai_use_sum + it_org_mixed * ai_use_sum + it_org_outsource * ai_use_sum + C(firm_size) + C(industry)',
    'variables': a_vars,
    'key_terms': ['it_org_internal:ai_use_sum', 'it_org_mixed:ai_use_sum', 'it_org_outsource:ai_use_sum'],
    'core_vif_vars': ['it_org_internal', 'it_org_mixed', 'it_org_outsource', 'ai_use_sum', 'it_org_internal:ai_use_sum', 'it_org_mixed:ai_use_sum', 'it_org_outsource:ai_use_sum'],
    'add_vars': '정보화 담당 체계 유형 더미',
    'use': use,
    'exclude_reason': '' if use else '변수 없음: ' + ', '.join(miss),
})

b_vars = ['effect_proc_improve', 'it_org_any', 'ai_use_sum', 'it_invest_sum', 'firm_size', 'industry']
use, miss = has_vars(b_vars)
model_specs.append({
    'model_name': 'B',
    'purpose': '정보화 투자 범위 추가 모형',
    'outcome': 'effect_proc_improve',
    'formula': 'effect_proc_improve ~ it_org_any * ai_use_sum + it_invest_sum + C(firm_size) + C(industry)',
    'variables': b_vars,
    'key_terms': ['it_org_any:ai_use_sum'],
    'core_vif_vars': ['it_org_any', 'ai_use_sum', 'it_org_any:ai_use_sum', 'it_invest_sum'],
    'add_vars': 'it_invest_sum',
    'use': use,
    'exclude_reason': '' if use else '변수 없음: ' + ', '.join(miss),
})

b2_vars = ['effect_proc_improve', 'it_org_any', 'ai_use_sum', 'it_invest_sum', 'it_invest_high', 'firm_size', 'industry']
use, miss = has_vars(b2_vars)
if use:
    # 값 체계와 결측이 안정적이면 B2 포함
    high_values = sorted(pd.Series(df['it_invest_high'].dropna().unique()).tolist())
    if not set(high_values).issubset({0, 1}):
        use = False
        miss = ['it_invest_high 값 체계 확인 필요: ' + str(high_values)]
model_specs.append({
    'model_name': 'B2',
    'purpose': '정보화 투자 범위 및 고투자 기업 추가 모형',
    'outcome': 'effect_proc_improve',
    'formula': 'effect_proc_improve ~ it_org_any * ai_use_sum + it_invest_sum + it_invest_high + C(firm_size) + C(industry)',
    'variables': b2_vars,
    'key_terms': ['it_org_any:ai_use_sum'],
    'core_vif_vars': ['it_org_any', 'ai_use_sum', 'it_org_any:ai_use_sum', 'it_invest_sum', 'it_invest_high'],
    'add_vars': 'it_invest_sum, it_invest_high',
    'use': use,
    'exclude_reason': '' if use else '제외: ' + ', '.join(miss),
})

c_vars = ['effect_proc_improve', 'it_org_any', 'ai_use_sum', 'ai_impl_sum', 'ai_purpose_sum', 'firm_size', 'industry']
use, miss = has_vars(c_vars)
model_specs.append({
    'model_name': 'C',
    'purpose': 'AI 구현 방식 및 활용 목적 추가 모형',
    'outcome': 'effect_proc_improve',
    'formula': 'effect_proc_improve ~ it_org_any * ai_use_sum + ai_impl_sum + ai_purpose_sum + C(firm_size) + C(industry)',
    'variables': c_vars,
    'key_terms': ['it_org_any:ai_use_sum'],
    'core_vif_vars': ['it_org_any', 'ai_use_sum', 'it_org_any:ai_use_sum', 'ai_impl_sum', 'ai_purpose_sum'],
    'add_vars': 'ai_impl_sum, ai_purpose_sum',
    'use': use,
    'exclude_reason': '' if use else '변수 없음: ' + ', '.join(miss),
})

d_vars = ['effect_proc_improve', 'it_org_any', 'ai_use_sum', 'it_invest_sum', 'ai_impl_sum', 'ai_purpose_sum', 'firm_size', 'industry']
d_formula = 'effect_proc_improve ~ it_org_any * ai_use_sum + it_invest_sum + ai_impl_sum + ai_purpose_sum + C(firm_size) + C(industry)'
d_add = 'it_invest_sum, ai_impl_sum, ai_purpose_sum'
d_vif = ['it_org_any', 'ai_use_sum', 'it_org_any:ai_use_sum', 'it_invest_sum', 'ai_impl_sum', 'ai_purpose_sum']
if 'it_invest_high' in df.columns and set(sorted(pd.Series(df['it_invest_high'].dropna().unique()).tolist())).issubset({0, 1}):
    d_vars = ['effect_proc_improve', 'it_org_any', 'ai_use_sum', 'it_invest_sum', 'it_invest_high', 'ai_impl_sum', 'ai_purpose_sum', 'firm_size', 'industry']
    d_formula = 'effect_proc_improve ~ it_org_any * ai_use_sum + it_invest_sum + it_invest_high + ai_impl_sum + ai_purpose_sum + C(firm_size) + C(industry)'
    d_add = 'it_invest_sum, it_invest_high, ai_impl_sum, ai_purpose_sum'
    d_vif = ['it_org_any', 'ai_use_sum', 'it_org_any:ai_use_sum', 'it_invest_sum', 'it_invest_high', 'ai_impl_sum', 'ai_purpose_sum']
use, miss = has_vars(d_vars)
model_specs.append({
    'model_name': 'D',
    'purpose': '전체 보강 변수 포함 모형',
    'outcome': 'effect_proc_improve',
    'formula': d_formula,
    'variables': d_vars,
    'key_terms': ['it_org_any:ai_use_sum'],
    'core_vif_vars': d_vif,
    'add_vars': d_add,
    'use': use,
    'exclude_reason': '' if use else '변수 없음: ' + ', '.join(miss),
})

design_rows = []
for spec in model_specs:
    design_rows.append({
        '모형명': spec['model_name'],
        '목적': spec['purpose'],
        '종속변수': spec['outcome'],
        '핵심 변수': ', '.join(spec['key_terms']),
        '추가 보강 변수': spec['add_vars'],
        '통제변수': 'C(firm_size), C(industry)',
        '사용 여부': '사용' if spec['use'] else '제외',
        '제외 사유': spec['exclude_reason'],
    })
design_summary = pd.DataFrame(design_rows)
print('[확장모형 설계 요약표]')
if display:
    display(design_summary)
else:
    print(design_summary.to_string(index=False))

model_n_check = []
for spec in model_specs:
    if spec['use']:
        sub = df[spec['variables']].dropna()
        used_n = len(sub)
        lost_n = total_n - used_n
        miss_by_var = {var: int(df[var].isna().sum()) for var in spec['variables']}
    else:
        used_n = np.nan
        lost_n = np.nan
        miss_by_var = {}
    model_n_check.append({'모형명': spec['model_name'], '사용 N': used_n, '전체 N 대비 제외 N': lost_n, '결측 변수별 N': miss_by_var})
model_n_check_df = pd.DataFrame(model_n_check)
print('\n[모형별 사용 N 확인]')
if display:
    display(model_n_check_df)
else:
    print(model_n_check_df.to_string(index=False))

[확장모형 설계 요약표]


,모형명,목적,종속변수,핵심 변수,추가 보강 변수,통제변수,사용 여부,제외 사유
0,A,정보화 담당 체계 유형별 확장모형,effect_proc_improve,"it_org_internal:ai_use_sum, it_org_mixed:ai_us...",정보화 담당 체계 유형 더미,"C(firm_size), C(industry)",사용,
1,B,정보화 투자 범위 추가 모형,effect_proc_improve,it_org_any:ai_use_sum,it_invest_sum,"C(firm_size), C(industry)",사용,
2,B2,정보화 투자 범위 및 고투자 기업 추가 모형,effect_proc_improve,it_org_any:ai_use_sum,"it_invest_sum, it_invest_high","C(firm_size), C(industry)",사용,
3,C,AI 구현 방식 및 활용 목적 추가 모형,effect_proc_improve,it_org_any:ai_use_sum,"ai_impl_sum, ai_purpose_sum","C(firm_size), C(industry)",사용,
4,D,전체 보강 변수 포함 모형,effect_proc_improve,it_org_any:ai_use_sum,"it_invest_sum, it_invest_high, ai_impl_sum, ai...","C(firm_size), C(industry)",사용,



[모형별 사용 N 확인]


,모형명,사용 N,전체 N 대비 제외 N,결측 변수별 N
0,A,12203,0,"{'effect_proc_improve': 0, 'it_org_internal': ..."
1,B,12203,0,"{'effect_proc_improve': 0, 'it_org_any': 0, 'a..."
2,B2,12203,0,"{'effect_proc_improve': 0, 'it_org_any': 0, 'a..."
3,C,12203,0,"{'effect_proc_improve': 0, 'it_org_any': 0, 'a..."
4,D,12203,0,"{'effect_proc_improve': 0, 'it_org_any': 0, 'a..."


## 5. 확장모형 A: 정보화 담당 체계 유형별 모형

단순 보유 여부 대신 `it_org_internal`, `it_org_mixed`, `it_org_outsource`와 AI 활용 수준의 상호작용항을 함께 넣은 유형별 탐색 모형이다.

In [4]:
model_results = {}
model_errors = {}

def fit_model(spec):
    model_data = df[spec['variables']].dropna().copy()
    result = smf.ols(spec['formula'], data=model_data).fit(cov_type='HC3')
    return result, len(model_data)

for spec in [s for s in model_specs if s['model_name'] == 'A']:
    if not spec['use']:
        model_errors[spec['model_name']] = spec['exclude_reason']
        print(f"모형 {spec['model_name']} 제외: {spec['exclude_reason']}")
        continue
    try:
        result, nobs = fit_model(spec)
        model_results[spec['model_name']] = {'result': result, 'n': nobs}
        print(f"===== Extension Model {spec['model_name']}: {spec['purpose']} =====")
        print(result.summary())
    except Exception as exc:
        model_errors[spec['model_name']] = repr(exc)
        print(f"모형 {spec['model_name']} 추정 실패: {repr(exc)}")

===== Extension Model A: 정보화 담당 체계 유형별 확장모형 =====
                             OLS Regression Results                            
Dep. Variable:     effect_proc_improve   R-squared:                       0.097
Model:                             OLS   Adj. R-squared:                  0.095
Method:                  Least Squares   F-statistic:                     62.07
Date:                 Sun, 03 May 2026   Prob (F-statistic):          2.18e-293
Time:                         03:00:31   Log-Likelihood:                -13414.
No. Observations:                12203   AIC:                         2.688e+04
Df Residuals:                    12177   BIC:                         2.707e+04
Df Model:                           25                                         
Covariance Type:                   HC3                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------

## 6. 확장모형 B: 정보화 투자 범위 추가 모형

정보화 투자 항목 개수(`it_invest_sum`)를 추가하고, `it_invest_high`가 안정적으로 존재할 경우 B2 모형을 추가로 추정한다.

In [5]:
for spec in [s for s in model_specs if s['model_name'] in ['B', 'B2']]:
    if not spec['use']:
        model_errors[spec['model_name']] = spec['exclude_reason']
        print(f"모형 {spec['model_name']} 제외: {spec['exclude_reason']}")
        continue
    try:
        result, nobs = fit_model(spec)
        model_results[spec['model_name']] = {'result': result, 'n': nobs}
        print(f"===== Extension Model {spec['model_name']}: {spec['purpose']} =====")
        print(result.summary())
    except Exception as exc:
        model_errors[spec['model_name']] = repr(exc)
        print(f"모형 {spec['model_name']} 추정 실패: {repr(exc)}")

===== Extension Model B: 정보화 투자 범위 추가 모형 =====
                             OLS Regression Results                            
Dep. Variable:     effect_proc_improve   R-squared:                       0.125
Model:                             OLS   Adj. R-squared:                  0.123
Method:                  Least Squares   F-statistic:                     89.20
Date:                 Sun, 03 May 2026   Prob (F-statistic):               0.00
Time:                         03:00:31   Log-Likelihood:                -13220.
No. Observations:                12203   AIC:                         2.649e+04
Df Residuals:                    12180   BIC:                         2.666e+04
Df Model:                           22                                         
Covariance Type:                   HC3                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------

## 7. 확장모형 C: AI 구현 방식 및 활용 목적 추가 모형

AI 구현 방식 개수(`ai_impl_sum`)와 AI 활용 목적 개수(`ai_purpose_sum`)를 추가해 핵심 상호작용항 방향이 유지되는지 확인한다.

In [6]:
for spec in [s for s in model_specs if s['model_name'] == 'C']:
    if not spec['use']:
        model_errors[spec['model_name']] = spec['exclude_reason']
        print(f"모형 {spec['model_name']} 제외: {spec['exclude_reason']}")
        continue
    try:
        result, nobs = fit_model(spec)
        model_results[spec['model_name']] = {'result': result, 'n': nobs}
        print(f"===== Extension Model {spec['model_name']}: {spec['purpose']} =====")
        print(result.summary())
    except Exception as exc:
        model_errors[spec['model_name']] = repr(exc)
        print(f"모형 {spec['model_name']} 추정 실패: {repr(exc)}")

===== Extension Model C: AI 구현 방식 및 활용 목적 추가 모형 =====
                             OLS Regression Results                            
Dep. Variable:     effect_proc_improve   R-squared:                       0.099
Model:                             OLS   Adj. R-squared:                  0.098
Method:                  Least Squares   F-statistic:                     63.40
Date:                 Sun, 03 May 2026   Prob (F-statistic):          2.61e-277
Time:                         03:00:31   Log-Likelihood:                -13397.
No. Observations:                12203   AIC:                         2.684e+04
Df Residuals:                    12179   BIC:                         2.702e+04
Df Model:                           23                                         
Covariance Type:                   HC3                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------

## 8. 확장모형 D: 전체 보강 변수 포함 모형

가능한 보강 변수를 함께 넣었을 때 핵심 상호작용항 방향이 유지되는지 탐색적으로 확인한다. 과통제 및 다중공선성 가능성이 있으므로 본문 주모형으로 사용하지 않는다.

In [7]:
for spec in [s for s in model_specs if s['model_name'] == 'D']:
    if not spec['use']:
        model_errors[spec['model_name']] = spec['exclude_reason']
        print(f"모형 {spec['model_name']} 제외: {spec['exclude_reason']}")
        continue
    try:
        result, nobs = fit_model(spec)
        model_results[spec['model_name']] = {'result': result, 'n': nobs}
        print(f"===== Extension Model {spec['model_name']}: {spec['purpose']} =====")
        print(result.summary())
    except Exception as exc:
        model_errors[spec['model_name']] = repr(exc)
        print(f"모형 {spec['model_name']} 추정 실패: {repr(exc)}")

===== Extension Model D: 전체 보강 변수 포함 모형 =====
                             OLS Regression Results                            
Dep. Variable:     effect_proc_improve   R-squared:                       0.126
Model:                             OLS   Adj. R-squared:                  0.124
Method:                  Least Squares   F-statistic:                     79.74
Date:                 Sun, 03 May 2026   Prob (F-statistic):               0.00
Time:                         03:00:31   Log-Likelihood:                -13213.
No. Observations:                12203   AIC:                         2.648e+04
Df Residuals:                    12177   BIC:                         2.667e+04
Df Model:                           25                                         
Covariance Type:                   HC3                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------

## 9. 확장모형 비교표

보고서용 요약표에는 핵심 확인 변수만 포함한다. A 모형은 세 유형별 상호작용항, B/B2/C/D 모형은 `it_org_any × ai_use_sum`을 중심으로 비교한다.

In [8]:
def stars(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''


def p_display(p):
    if pd.isna(p):
        return 'NA'
    if p < 0.001:
        return '<.001'
    return f'{p:.3f}'


def md_table(data: pd.DataFrame) -> str:
    def fmt(value):
        if pd.isna(value):
            return 'NA'
        if isinstance(value, (float, np.floating)):
            return f'{float(value):.3f}'
        return str(value).replace('|', '\\|').replace('\n', '<br>')
    cols = list(data.columns)
    lines = ['| ' + ' | '.join(cols) + ' |']
    lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
    for _, row in data.iterrows():
        lines.append('| ' + ' | '.join(fmt(row[col]) for col in cols) + ' |')
    return '\n'.join(lines)


def save_table(data: pd.DataFrame, path: Path, **kwargs):
    if SAVE_OUTPUTS:
        data.to_csv(path, **kwargs)
    return path


def save_text(text: str, path: Path):
    if SAVE_OUTPUTS:
        path.write_text(text, encoding='utf-8')
    return path


def print_save_status(path: Path):
    if SAVE_OUTPUTS:
        print('-', path.relative_to(BASE_DIR))
    else:
        print(f'- {path.name}: 저장 비활성화, 노트북 출력만 확인')

term_labels = {
    'it_org_any': '정보화 담당 체계 보유',
    'it_org_internal': '내부 전담 조직/인력',
    'it_org_mixed': '겸직 인력',
    'it_org_outsource': '외부 위탁',
    'ai_use_sum': 'AI 활용 수준',
    'it_org_any:ai_use_sum': '정보화 담당 체계 × AI 활용 수준',
    'it_org_internal:ai_use_sum': '내부 전담 조직/인력 × AI 활용 수준',
    'it_org_mixed:ai_use_sum': '겸직 인력 × AI 활용 수준',
    'it_org_outsource:ai_use_sum': '외부 위탁 × AI 활용 수준',
    'it_invest_sum': '정보화 투자 항목 개수',
    'it_invest_high': '정보화 고투자 기업',
    'ai_impl_sum': 'AI 구현 방식 개수',
    'ai_purpose_sum': 'AI 활용 목적 개수',
}

summary_rows = []
full_rows = []

for spec in model_specs:
    if spec['model_name'] in model_results:
        res = model_results[spec['model_name']]['result']
        nobs = model_results[spec['model_name']]['n']
        conf = res.conf_int(alpha=0.05)
        # full coefficient table
        for term in res.params.index:
            full_rows.append({
                'model_name': spec['model_name'],
                'term': term,
                'term_label': term_labels.get(term, term),
                'coef': res.params[term],
                'robust_se': res.bse[term],
                't_value': res.tvalues[term],
                'p_value': res.pvalues[term],
                'significance': stars(res.pvalues[term]),
                'ci_lower': conf.loc[term, 0],
                'ci_upper': conf.loc[term, 1],
                'n': int(nobs),
                'r_squared': res.rsquared,
                'adj_r_squared': res.rsquared_adj,
            })
        # summary key terms
        for term in spec['key_terms']:
            if term in res.params.index:
                coef = res.params[term]
                pval = res.pvalues[term]
                direction = '일치' if coef > 0 else '불일치'
                sig_consistent = '일치' if pval < 0.05 else '불일치'
                memo = '핵심 상호작용항이 양(+) 방향이며 유의' if coef > 0 and pval < 0.05 else '방향 또는 유의성 확인 필요'
                summary_rows.append({
                    '모형': spec['model_name'],
                    '목적': spec['purpose'],
                    '핵심 확인 변수': term_labels.get(term, term),
                    '계수': coef,
                    'Robust SE': res.bse[term],
                    'p-value': pval,
                    '유의성': stars(pval),
                    'N': int(nobs),
                    'R-squared': res.rsquared,
                    'Adjusted R-squared': res.rsquared_adj,
                    '주모형과 방향 일치 여부': direction,
                    '주모형과 유의성 일치 여부': sig_consistent,
                    '해석 메모': memo,
                })
            else:
                summary_rows.append({
                    '모형': spec['model_name'],
                    '목적': spec['purpose'],
                    '핵심 확인 변수': term_labels.get(term, term),
                    '계수': np.nan,
                    'Robust SE': np.nan,
                    'p-value': np.nan,
                    '유의성': '',
                    'N': nobs,
                    'R-squared': np.nan,
                    'Adjusted R-squared': np.nan,
                    '주모형과 방향 일치 여부': '확인 필요',
                    '주모형과 유의성 일치 여부': '확인 필요',
                    '해석 메모': '해당 term을 결과에서 찾지 못함',
                })
    else:
        reason = model_errors.get(spec['model_name'], spec['exclude_reason'] or '추정 결과 없음')
        for term in spec['key_terms']:
            summary_rows.append({
                '모형': spec['model_name'],
                '목적': spec['purpose'],
                '핵심 확인 변수': term_labels.get(term, term),
                '계수': np.nan,
                'Robust SE': np.nan,
                'p-value': np.nan,
                '유의성': '',
                'N': np.nan,
                'R-squared': np.nan,
                'Adjusted R-squared': np.nan,
                '주모형과 방향 일치 여부': '확인 필요',
                '주모형과 유의성 일치 여부': '확인 필요',
                '해석 메모': '모형 제외 또는 추정 실패: ' + reason,
            })

summary_table = pd.DataFrame(summary_rows)
full_coef_table = pd.DataFrame(full_rows)

for col in ['계수', 'Robust SE', 'p-value', 'R-squared', 'Adjusted R-squared']:
    summary_table[col] = summary_table[col].astype(float).round(3)
for col in ['coef', 'robust_se', 't_value', 'p_value', 'ci_lower', 'ci_upper', 'r_squared', 'adj_r_squared']:
    if col in full_coef_table.columns:
        full_coef_table[col] = full_coef_table[col].astype(float).round(3)

summary_display = summary_table.copy()
summary_display['p-value'] = summary_display['p-value'].apply(p_display)

print('[Table A2. 확장모형 요약표]')
if display:
    display(summary_display)
else:
    print(summary_display.to_string(index=False))

[Table A2. 확장모형 요약표]


,모형,목적,핵심 확인 변수,계수,Robust SE,p-value,유의성,N,R-squared,Adjusted R-squared,주모형과 방향 일치 여부,주모형과 유의성 일치 여부,해석 메모
0,A,정보화 담당 체계 유형별 확장모형,내부 전담 조직/인력 × AI 활용 수준,0.020,0.011,0.074,,12203,0.097,0.095,일치,불일치,방향 또는 유의성 확인 필요
1,A,정보화 담당 체계 유형별 확장모형,겸직 인력 × AI 활용 수준,0.023,0.012,0.050,*,12203,0.097,0.095,일치,일치,핵심 상호작용항이 양(+) 방향이며 유의
2,A,정보화 담당 체계 유형별 확장모형,외부 위탁 × AI 활용 수준,0.009,0.014,0.516,,12203,0.097,0.095,일치,불일치,방향 또는 유의성 확인 필요
3,B,정보화 투자 범위 추가 모형,정보화 담당 체계 × AI 활용 수준,0.057,0.011,<.001,***,12203,0.125,0.123,일치,일치,핵심 상호작용항이 양(+) 방향이며 유의
4,B2,정보화 투자 범위 및 고투자 기업 추가 모형,정보화 담당 체계 × AI 활용 수준,0.057,0.011,<.001,***,12203,0.125,0.124,일치,일치,핵심 상호작용항이 양(+) 방향이며 유의
5,C,AI 구현 방식 및 활용 목적 추가 모형,정보화 담당 체계 × AI 활용 수준,0.066,0.011,<.001,***,12203,0.099,0.098,일치,일치,핵심 상호작용항이 양(+) 방향이며 유의
6,D,전체 보강 변수 포함 모형,정보화 담당 체계 × AI 활용 수준,0.059,0.011,<.001,***,12203,0.126,0.124,일치,일치,핵심 상호작용항이 양(+) 방향이며 유의


## 10. 부록용 전체 계수표

각 확장모형의 전체 계수를 long/tidy 형태로 정리한다. firm_size와 industry 더미 계수는 이 부록용 전체 계수표에 포함된다.

In [9]:
print('[Table A3. 부록용 전체 계수표]')
if display:
    display(full_coef_table)
else:
    print(full_coef_table.to_string(index=False))

# VIF 진단표
vif_rows = []

def build_vif_frame(spec):
    model_data = df[spec['variables']].dropna().copy()
    X = pd.DataFrame(index=model_data.index)
    for var in spec['core_vif_vars']:
        if ':' in var:
            left, right = var.split(':')
            if left in model_data.columns and right in model_data.columns:
                X[var] = model_data[left].astype(float) * model_data[right].astype(float)
        elif var in model_data.columns:
            X[var] = model_data[var].astype(float)
    return X.dropna()

for spec in model_specs:
    if not spec['use'] or spec['model_name'] not in model_results:
        for var in spec['core_vif_vars']:
            vif_rows.append({'model_name': spec['model_name'], 'variable': var, 'vif': np.nan, 'note': '모형 제외 또는 추정 실패'})
        continue
    try:
        X = build_vif_frame(spec)
        X = sm.add_constant(X, has_constant='add')
        for i, col in enumerate(X.columns):
            if col == 'const':
                continue
            vif = variance_inflation_factor(X.values, i)
            note = '높음' if vif >= 10 else '주의' if vif >= 5 else '양호'
            vif_rows.append({'model_name': spec['model_name'], 'variable': col, 'vif': vif, 'note': note})
    except Exception as exc:
        for var in spec['core_vif_vars']:
            vif_rows.append({'model_name': spec['model_name'], 'variable': var, 'vif': np.nan, 'note': 'VIF 계산 실패: ' + repr(exc)})

vif_table = pd.DataFrame(vif_rows)
vif_table['vif'] = vif_table['vif'].astype(float).round(3)
print('\n[Table A4. VIF 진단표]')
if display:
    display(vif_table)
else:
    print(vif_table.to_string(index=False))

[Table A3. 부록용 전체 계수표]


,model_name,term,term_label,coef,robust_se,t_value,p_value,significance,ci_lower,ci_upper,n,r_squared,adj_r_squared
0,A,Intercept,Intercept,3.731,0.034,110.014,0.000,***,3.665,3.798,12203,0.097,0.095
1,A,C(firm_size)[T.2],C(firm_size)[T.2],0.095,0.016,5.921,0.000,***,0.064,0.127,12203,0.097,0.095
2,A,C(firm_size)[T.3],C(firm_size)[T.3],0.231,0.021,10.939,0.000,***,0.190,0.273,12203,0.097,0.095
3,A,C(firm_size)[T.4],C(firm_size)[T.4],0.279,0.037,7.540,0.000,***,0.206,0.351,12203,0.097,0.095
4,A,C(industry)[T.2],C(industry)[T.2],-0.019,0.036,-0.515,0.607,,-0.090,0.052,12203,0.097,0.095
...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,D,it_org_any:ai_use_sum,정보화 담당 체계 × AI 활용 수준,0.059,0.011,5.395,0.000,***,0.037,0.080,12203,0.126,0.124
119,D,it_invest_sum,정보화 투자 항목 개수,0.114,0.006,19.341,0.000,***,0.102,0.125,12203,0.126,0.124
120,D,it_invest_high,정보화 고투자 기업,-0.049,0.023,-2.102,0.036,*,-0.095,-0.003,12203,0.126,0.124
121,D,ai_impl_sum,AI 구현 방식 개수,-0.002,0.016,-0.134,0.894,,-0.033,0.029,12203,0.126,0.124



[Table A4. VIF 진단표]


,model_name,variable,vif,note
0,A,it_org_internal,1.819,양호
1,A,it_org_mixed,1.496,양호
2,A,it_org_outsource,1.467,양호
3,A,ai_use_sum,2.136,양호
4,A,it_org_internal:ai_use_sum,2.449,양호
5,A,it_org_mixed:ai_use_sum,2.037,양호
6,A,it_org_outsource:ai_use_sum,1.694,양호
7,B,it_org_any,1.758,양호
8,B,ai_use_sum,2.740,양호
9,B,it_org_any:ai_use_sum,3.274,양호


## 11. 보고서용 요약 문단

확장모형 결과를 5~8문장으로 요약한다. 본문 주모형을 대체하지 않는 부록용 탐색 분석이라는 점을 명시한다.

In [10]:
def get_summary(model, label_contains=None):
    sub = summary_table[summary_table['모형'] == model]
    if label_contains is not None:
        sub = sub[sub['핵심 확인 변수'].str.contains(label_contains, regex=False)]
    return sub.iloc[0] if len(sub) else None

row_a_internal = get_summary('A', '내부 전담')
row_a_mixed = get_summary('A', '겸직')
row_a_outsource = get_summary('A', '외부 위탁')
row_b = get_summary('B')
row_b2 = get_summary('B2')
row_c = get_summary('C')
row_d = get_summary('D')

positive_a = []
for label, row in [('내부 전담 조직/인력', row_a_internal), ('겸직 인력', row_a_mixed), ('외부 위탁', row_a_outsource)]:
    if row is not None and not pd.isna(row['계수']) and row['계수'] > 0:
        positive_a.append(f'{label}({row["계수"]:.3f}{row["유의성"]})')

sentences = []
sentences.append('교수님 피드백을 반영하여 정보화 담당 체계의 세부 유형, 정보화 투자 범위, AI 구현 방식 및 활용 목적 변수를 추가한 확장모형을 추정하였다.')
if positive_a:
    sentences.append('유형별 모형 A에서는 ' + ', '.join(positive_a) + '의 상호작용항이 양(+)의 방향으로 나타났다.')
else:
    sentences.append('유형별 모형 A에서는 양(+) 방향의 상호작용항이 뚜렷하게 확인되지 않았다.')
if row_b is not None and not pd.isna(row_b['계수']):
    sentences.append(f'정보화 투자 범위를 추가한 모형 B에서도 it_org_any × ai_use_sum 계수는 {row_b["계수"]:.3f}{row_b["유의성"]}로 나타나 주모형과 동일한 방향을 보였다.')
if row_b2 is not None and not pd.isna(row_b2['계수']):
    sentences.append(f'it_invest_high를 추가한 B2에서도 핵심 상호작용항 계수는 {row_b2["계수"]:.3f}{row_b2["유의성"]}로 나타났다.')
if row_c is not None and not pd.isna(row_c['계수']):
    sentences.append(f'AI 구현 방식과 활용 목적을 추가한 모형 C에서도 핵심 상호작용항 계수는 {row_c["계수"]:.3f}{row_c["유의성"]}로 나타났다.')
if row_d is not None and not pd.isna(row_d['계수']):
    d_text = '유지되었다' if row_d['계수'] > 0 else '유지되지 않았다'
    sentences.append(f'전체 보강 변수를 포함한 모형 D에서도 핵심 계수는 {row_d["계수"]:.3f}{row_d["유의성"]}로 나타나 방향은 {d_text}.')
sentences.append('다만 확장모형은 변수 간 개념적 중첩과 다중공선성 가능성이 있으므로, 본문 주모형을 대체하기보다는 부록용 보조분석으로 해석하는 것이 적절하다.')
sentences.append('따라서 본문에서는 주모형을 중심으로 핵심 결과만 요약하고, 전체 보강 결과는 부록에 제시한다.')

print('[보고서용 요약 문단]')
for s in sentences:
    print('-', s)

[보고서용 요약 문단]
- 교수님 피드백을 반영하여 정보화 담당 체계의 세부 유형, 정보화 투자 범위, AI 구현 방식 및 활용 목적 변수를 추가한 확장모형을 추정하였다.
- 유형별 모형 A에서는 내부 전담 조직/인력(0.020), 겸직 인력(0.023*), 외부 위탁(0.009)의 상호작용항이 양(+)의 방향으로 나타났다.
- 정보화 투자 범위를 추가한 모형 B에서도 it_org_any × ai_use_sum 계수는 0.057***로 나타나 주모형과 동일한 방향을 보였다.
- it_invest_high를 추가한 B2에서도 핵심 상호작용항 계수는 0.057***로 나타났다.
- AI 구현 방식과 활용 목적을 추가한 모형 C에서도 핵심 상호작용항 계수는 0.066***로 나타났다.
- 전체 보강 변수를 포함한 모형 D에서도 핵심 계수는 0.059***로 나타나 방향은 유지되었다.
- 다만 확장모형은 변수 간 개념적 중첩과 다중공선성 가능성이 있으므로, 본문 주모형을 대체하기보다는 부록용 보조분석으로 해석하는 것이 적절하다.
- 따라서 본문에서는 주모형을 중심으로 핵심 결과만 요약하고, 전체 보강 결과는 부록에 제시한다.


## 12. 확인 필요 사항

저장 여부, 저장 파일 목록, 모형 실패 여부, VIF 주의 항목, 모형별 사용 N을 확인한다.

In [11]:
design_csv = OUT_DIR / 'tableA1_extended_model_design.csv'
summary_csv = OUT_DIR / 'tableA2_extended_model_summary.csv'
summary_md = OUT_DIR / 'tableA2_extended_model_summary.md'
full_csv = OUT_DIR / 'tableA3_extended_model_full_coefficients.csv'
vif_csv = OUT_DIR / 'tableA4_extended_model_vif.csv'
interp_md = OUT_DIR / 'extended_model_interpretation.md'

save_table(design_summary, design_csv, index=False, encoding='utf-8-sig')
save_table(summary_table, summary_csv, index=False, encoding='utf-8-sig')
save_text('# Table A2. 확장모형 요약표\n\n' + md_table(summary_display) + '\n', summary_md)
save_table(full_coef_table, full_csv, index=False, encoding='utf-8-sig')
save_table(vif_table, vif_csv, index=False, encoding='utf-8-sig')
save_text('# 확장모형 보고서용 요약 문단\n\n' + f'- 사용 데이터: `{DATA_PATH.relative_to(BASE_DIR)}`\n- 전체 N: {total_n:,}\n\n' + '\n'.join(f'- {s}' for s in sentences) + '\n', interp_md)

need_check = []
if missing_candidates:
    need_check.append('존재하지 않아 제외 또는 확인이 필요한 후보 변수: ' + ', '.join(missing_candidates))
else:
    need_check.append('확장모형 후보 변수와 참고 변수가 모두 존재함')

for spec in model_specs:
    if not spec['use']:
        need_check.append(f"모형 {spec['model_name']} 제외: {spec['exclude_reason']}")

if model_errors:
    for model_name, err in model_errors.items():
        need_check.append(f'모형 {model_name} 추정 확인 필요: {err}')
else:
    need_check.append('모든 사용 확장모형이 오류 없이 추정됨')

if all(row['전체 N 대비 제외 N'] == 0 for _, row in model_n_check_df.dropna(subset=['전체 N 대비 제외 N']).iterrows()):
    need_check.append('모든 사용 확장모형에서 사용 N이 전체 N과 동일하며, 결측으로 인한 표본 손실 없음')
else:
    for _, row in model_n_check_df.iterrows():
        if pd.notna(row['전체 N 대비 제외 N']) and row['전체 N 대비 제외 N'] > 0:
            need_check.append(f"모형 {row['모형명']}에서 전체 N 대비 {row['전체 N 대비 제외 N']}건 제외됨: {row['결측 변수별 N']}")

vif_attention = vif_table[vif_table['note'].isin(['주의', '높음'])]
if len(vif_attention):
    need_check.append('VIF 주의/높음 항목 존재: ' + '; '.join(f"{r['model_name']}:{r['variable']}={r['vif']:.3f}({r['note']})" for _, r in vif_attention.iterrows()))
else:
    need_check.append('핵심 변수 VIF 기준에서 5 이상 항목 없음')
need_check.append('확장모형은 과통제 및 개념적 중첩 가능성이 있으므로 부록용 탐색 분석으로 제한함')

print('1. 사용한 데이터 파일명과 전체 N')
print('-', DATA_PATH.relative_to(BASE_DIR))
print('-', f'N = {total_n:,}')
print('SAVE_OUTPUTS:', SAVE_OUTPUTS)

print('\n2. 변수 존재 여부 및 결측 확인표')
if display:
    display(variable_check)
else:
    print(variable_check.to_string(index=False))

print('\n3. 확장모형 설계 요약표')
if display:
    display(design_summary)
else:
    print(design_summary.to_string(index=False))

print('\n5. Table A2. 확장모형 요약표')
if display:
    display(summary_display)
else:
    print(summary_display.to_string(index=False))

print('\n6. Table A3. 부록용 전체 계수표')
if display:
    display(full_coef_table)
else:
    print(full_coef_table.to_string(index=False))

print('\n7. Table A4. VIF 진단표')
if display:
    display(vif_table)
else:
    print(vif_table.to_string(index=False))

print('\n8. 보고서용 요약 문단')
for s in sentences:
    print('-', s)

print('\n9. 저장 여부 및 저장 파일 목록')
if SAVE_OUTPUTS:
    print('저장 완료')
else:
    print('저장 비활성화: 파일은 생성하지 않고 노트북 화면에만 표시했습니다.')
for path in [design_csv, summary_csv, summary_md, full_csv, vif_csv, interp_md]:
    print_save_status(path)

print('\n10. 확인 필요 사항')
for note in need_check:
    print('-', note)

1. 사용한 데이터 파일명과 전체 N
- working/analysis/nia_2024_analysis_total.csv
- N = 12,203
SAVE_OUTPUTS: False

2. 변수 존재 여부 및 결측 확인표


,변수명,분류,존재 여부,유효 N,결측 N,고유값 예시
0,effect_proc_improve,필수,True,12203,0,"3, 4, 5, 2, 1"
1,it_org_any,필수,True,12203,0,"0, 1"
2,ai_use_sum,필수,True,12203,0,"1, 0, 2, 3, 4, 5, 6, 7"
3,firm_size,필수,True,12203,0,"1, 2, 3, 4"
4,industry,필수,True,12203,0,"1, 2, 3, 4, 5, 6, 7, 8"
5,it_org_internal,A 후보,True,12203,0,"0, 1"
6,it_org_mixed,A 후보,True,12203,0,"0, 1"
7,it_org_outsource,A 후보,True,12203,0,"0, 1"
8,it_invest_sum,B 후보,True,12203,0,"1, 2, 3, 4, 5, 6, 7, 8"
9,it_invest_share,B 후보,True,12203,0,"1.0, 4.0, 65.0, 2.0, 70.0, 35.0, 8.0, 50.0"



3. 확장모형 설계 요약표


,모형명,목적,종속변수,핵심 변수,추가 보강 변수,통제변수,사용 여부,제외 사유
0,A,정보화 담당 체계 유형별 확장모형,effect_proc_improve,"it_org_internal:ai_use_sum, it_org_mixed:ai_us...",정보화 담당 체계 유형 더미,"C(firm_size), C(industry)",사용,
1,B,정보화 투자 범위 추가 모형,effect_proc_improve,it_org_any:ai_use_sum,it_invest_sum,"C(firm_size), C(industry)",사용,
2,B2,정보화 투자 범위 및 고투자 기업 추가 모형,effect_proc_improve,it_org_any:ai_use_sum,"it_invest_sum, it_invest_high","C(firm_size), C(industry)",사용,
3,C,AI 구현 방식 및 활용 목적 추가 모형,effect_proc_improve,it_org_any:ai_use_sum,"ai_impl_sum, ai_purpose_sum","C(firm_size), C(industry)",사용,
4,D,전체 보강 변수 포함 모형,effect_proc_improve,it_org_any:ai_use_sum,"it_invest_sum, it_invest_high, ai_impl_sum, ai...","C(firm_size), C(industry)",사용,



5. Table A2. 확장모형 요약표


...


6. Table A3. 부록용 전체 계수표


,model_name,term,term_label,coef,robust_se,t_value,p_value,significance,ci_lower,ci_upper,n,r_squared,adj_r_squared
0,A,Intercept,Intercept,3.731,0.034,110.014,0.000,***,3.665,3.798,12203,0.097,0.095
1,A,C(firm_size)[T.2],C(firm_size)[T.2],0.095,0.016,5.921,0.000,***,0.064,0.127,12203,0.097,0.095
2,A,C(firm_size)[T.3],C(firm_size)[T.3],0.231,0.021,10.939,0.000,***,0.190,0.273,12203,0.097,0.095
3,A,C(firm_size)[T.4],C(firm_size)[T.4],0.279,0.037,7.540,0.000,***,0.206,0.351,12203,0.097,0.095
4,A,C(industry)[T.2],C(industry)[T.2],-0.019,0.036,-0.515,0.607,,-0.090,0.052,12203,0.097,0.095
...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,D,it_org_any:ai_use_sum,정보화 담당 체계 × AI 활용 수준,0.059,0.011,5.395,0.000,***,0.037,0.080,12203,0.126,0.124
119,D,it_invest_sum,정보화 투자 항목 개수,0.114,0.006,19.341,0.000,***,0.102,0.125,12203,0.126,0.124
120,D,it_invest_high,정보화 고투자 기업,-0.049,0.023,-2.102,0.036,*,-0.095,-0.003,12203,0.126,0.124
121,D,ai_impl_sum,AI 구현 방식 개수,-0.002,0.016,-0.134,0.894,,-0.033,0.029,12203,0.126,0.124



7. Table A4. VIF 진단표


,model_name,variable,vif,note
0,A,it_org_internal,1.819,양호
1,A,it_org_mixed,1.496,양호
2,A,it_org_outsource,1.467,양호
3,A,ai_use_sum,2.136,양호
4,A,it_org_internal:ai_use_sum,2.449,양호
5,A,it_org_mixed:ai_use_sum,2.037,양호
6,A,it_org_outsource:ai_use_sum,1.694,양호
7,B,it_org_any,1.758,양호
8,B,ai_use_sum,2.740,양호
9,B,it_org_any:ai_use_sum,3.274,양호



8. 보고서용 요약 문단
- 교수님 피드백을 반영하여 정보화 담당 체계의 세부 유형, 정보화 투자 범위, AI 구현 방식 및 활용 목적 변수를 추가한 확장모형을 추정하였다.
- 유형별 모형 A에서는 내부 전담 조직/인력(0.020), 겸직 인력(0.023*), 외부 위탁(0.009)의 상호작용항이 양(+)의 방향으로 나타났다.
- 정보화 투자 범위를 추가한 모형 B에서도 it_org_any × ai_use_sum 계수는 0.057***로 나타나 주모형과 동일한 방향을 보였다.
- it_invest_high를 추가한 B2에서도 핵심 상호작용항 계수는 0.057***로 나타났다.
- AI 구현 방식과 활용 목적을 추가한 모형 C에서도 핵심 상호작용항 계수는 0.066***로 나타났다.
- 전체 보강 변수를 포함한 모형 D에서도 핵심 계수는 0.059***로 나타나 방향은 유지되었다.
- 다만 확장모형은 변수 간 개념적 중첩과 다중공선성 가능성이 있으므로, 본문 주모형을 대체하기보다는 부록용 보조분석으로 해석하는 것이 적절하다.
- 따라서 본문에서는 주모형을 중심으로 핵심 결과만 요약하고, 전체 보강 결과는 부록에 제시한다.

9. 저장 여부 및 저장 파일 목록
저장 비활성화: 파일은 생성하지 않고 노트북 화면에만 표시했습니다.
- tableA1_extended_model_design.csv: 저장 비활성화, 노트북 출력만 확인
- tableA2_extended_model_summary.csv: 저장 비활성화, 노트북 출력만 확인
- tableA2_extended_model_summary.md: 저장 비활성화, 노트북 출력만 확인
- tableA3_extended_model_full_coefficients.csv: 저장 비활성화, 노트북 출력만 확인
- tableA4_extended_model_vif.csv: 저장 비활성화, 노트북 출력만 확인
- extended_model_interpretation.md: 저장 비활성화, 노트북 출력만 확인

10. 확인 필요 